In [17]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import gradio as gr
import os

# ========== 1. 讀取資料、型態處理 ==========
def load_and_merge_data():
    winners = pd.read_csv('./data/winners.csv')
    drivers = pd.read_csv('./data/drivers_updated.csv')
    teams = pd.read_csv('./data/teams_updated.csv')
    winners['year'] = pd.to_datetime(winners['Date']).dt.year.astype(int)
    for df_ in [winners, drivers, teams]:
        for col in df_.columns:
            df_[col] = df_[col].astype(str).str.strip()
    winners['year'] = winners['year'].astype(int)
    drivers['year'] = drivers['year'].astype(int)
    teams['year'] = teams['year'].astype(int)
    df = winners.merge(
        drivers[['Driver', 'Nationality', 'Car', 'year']],
        left_on=['Winner', 'Car', 'year'],
        right_on=['Driver', 'Car', 'year'],
        how='left'
    )
    df = df.rename(columns={'Car': 'Team'})
    df = df[['year', 'Grand Prix', 'Winner', 'Team', 'Driver', 'Nationality']]
    for col in ['year', 'Grand Prix', 'Winner', 'Team', 'Driver', 'Nationality']:
        df[col] = df[col].astype(str).str.strip()
    df = df.dropna().reset_index(drop=True)
    return df, drivers

df, drivers = load_and_merge_data()

# ========== 2. 只保留出現兩次以上的冠軍 ==========
vc = df['Winner'].value_counts()
multi_winner = vc[vc >= 2].index
df = df[df['Winner'].isin(multi_winner)].reset_index(drop=True)

for col in ['year', 'Grand Prix', 'Winner', 'Team', 'Driver', 'Nationality']:
    df[col] = df[col].astype(str).str.strip()

# ========== 3. 避免資料洩漏，分組 ==========
df['race_id'] = df['year'].astype(str) + "_" + df['Grand Prix']
unique_race_ids = df['race_id'].unique()
train_ids, test_ids = train_test_split(unique_race_ids, test_size=0.2, random_state=42)
train_df = df[df['race_id'].isin(train_ids)].reset_index(drop=True)
test_df = df[df['race_id'].isin(test_ids)].reset_index(drop=True)

# ========== 4. Label Encoding ==========
cat_cols = ['Grand Prix', 'Team', 'Driver', 'Nationality']
num_cols = ['year']
target_col = 'Winner'
label_encoders = {col: LabelEncoder().fit(df[col]) for col in cat_cols}
for ddf in [train_df, test_df]:
    for col, le in label_encoders.items():
        ddf[col] = le.transform(ddf[col])
scaler = StandardScaler()
for ddf in [train_df, test_df]:
    ddf[num_cols] = scaler.fit_transform(ddf[num_cols])
target_le = LabelEncoder().fit(df[target_col])
train_df['target'] = target_le.transform(train_df[target_col])
test_df['target'] = target_le.transform(test_df[target_col])

# ========== 5. Dataset & DataLoader ==========
class F1Dataset(Dataset):
    def __init__(self, df):
        self.cat = df[cat_cols].values
        self.num = df[num_cols].values
        self.y = df['target'].values
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return torch.tensor(self.cat[idx], dtype=torch.long), \
               torch.tensor(self.num[idx], dtype=torch.float32), \
               torch.tensor(self.y[idx], dtype=torch.long)

batch_size = 64
trainset = F1Dataset(train_df)
testset = F1Dataset(test_df)

# ----- Balanced Sampling -----
class_sample_count = np.array([len(np.where(train_df['target']==t)[0]) for t in np.unique(train_df['target'])])
weight = 1. / class_sample_count
samples_weight = np.array([weight[t] for t in train_df['target']])
samples_weight = torch.from_numpy(samples_weight).float()
sampler = WeightedRandomSampler(samples_weight, len(samples_weight))
trainloader = DataLoader(trainset, batch_size=batch_size, sampler=sampler)
testloader = DataLoader(testset, batch_size=batch_size)

# ========== 6. DNN+Embedding 模型 ==========
class F1DNN(nn.Module):
    def __init__(self, cat_dims, num_num, emb_dim=8, hidden_dim=128, num_classes=20):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(dim, emb_dim) for dim in cat_dims])
        self.bn_num = nn.BatchNorm1d(num_num)
        self.fc1 = nn.Linear(len(cat_cols)*emb_dim + num_num, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)  # 增加 Dropout
        self.fc2 = nn.Linear(hidden_dim, num_classes)
    def forward(self, x_cat, x_num):
        x_emb = torch.cat([emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)], dim=1)
        x_num = self.bn_num(x_num)
        x = torch.cat([x_emb, x_num], dim=1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

cat_dims = [len(le.classes_) for le in label_encoders.values()]
num_classes = len(target_le.classes_)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = F1DNN(cat_dims, len(num_cols), emb_dim=8, hidden_dim=128, num_classes=num_classes).to(device)

# ========== 7. 訓練 ==========
def train_model(model, trainloader, n_epoch=50, lr=0.0005, patience=8):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    best_loss = np.inf
    no_improve = 0
    for epoch in range(n_epoch):
        model.train()
        total_loss = 0
        for x_cat, x_num, y in trainloader:
            x_cat, x_num, y = x_cat.to(device), x_num.to(device), y.to(device)
            logits = model(x_cat, x_num)
            loss = criterion(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(trainloader)
        print(f"Epoch {epoch+1}/{n_epoch} | Train Loss: {avg_loss:.4f}")
        if avg_loss < best_loss:
            best_loss = avg_loss
            no_improve = 0
            torch.save(model.state_dict(), './data/f1_dnn_embedding.pth')
        else:
            no_improve += 1
            if no_improve >= patience:
                print("Early stopping triggered!")
                break

# ========== 8. 驗證 ==========
def eval_model(model, testloader):
    model.eval()
    all_y, all_pred = [], []
    with torch.no_grad():
        for x_cat, x_num, y in testloader:
            x_cat, x_num = x_cat.to(device), x_num.to(device)
            logits = model(x_cat, x_num)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_pred.extend(preds)
            all_y.extend(y.numpy())
    acc = accuracy_score(all_y, all_pred)
    print(f"Test Accuracy: {acc:.4f}")
    labels = np.unique(np.concatenate([all_y, all_pred]))
    target_names = target_le.inverse_transform(labels)
    print(classification_report(all_y, all_pred, labels=labels, target_names=target_names))
    return acc

# ========== 9. 訓練＆驗證 ==========
train_model(model, trainloader, n_epoch=50, lr=0.0005, patience=8)
model.load_state_dict(torch.load('./data/f1_dnn_embedding.pth'))
eval_model(model, testloader)

# ========== 10. 全年份/場地/車手對照 ==========
drivers['year'] = drivers['year'].astype(int)
drivers_year_dict = {}
teams_year_dict = {}
driver_team_nat_dict = {}
for y in sorted(drivers['year'].unique()):
    drivers_this_year = drivers[drivers['year']==y]
    driver_names = sorted(drivers_this_year['Driver'].dropna().astype(str).unique())
    drivers_year_dict[y] = driver_names
    teams_year_dict[y] = sorted(drivers_this_year['Car'].dropna().astype(str).unique())
    driver_team_nat_dict[y] = {}
    for _, row in drivers_this_year.iterrows():
        driver_team_nat_dict[y][row['Driver']] = (row['Car'], row['Nationality'])

# ========== 11. 年份與場地選單 ==========
all_years = sorted(df['year'].astype(int).unique())
all_grandprix = sorted(df['Grand Prix'].unique())

# ========== 12. Gradio預測 ==========
def gradio_predict(year, grand_prix):
    year = int(year)
    if year not in drivers_year_dict:
        return "該年份查無參賽車手資料！"
    driver_names = drivers_year_dict[year]
    driver_team_nat = driver_team_nat_dict[year]
    all_results = []
    for driver in driver_names:
        team, nationality = driver_team_nat.get(driver, ("", ""))
        input_dict = {
            'year': float(year),
            'Grand Prix': grand_prix,
            'Team': team,
            'Driver': driver,
            'Nationality': nationality
        }
        x_cat = []
        for col in cat_cols:
            le = label_encoders[col]
            if input_dict[col] in le.classes_:
                idx = le.transform([input_dict[col]])[0]
            else:
                idx = 0
            x_cat.append(idx)
        x_cat = torch.tensor([x_cat], dtype=torch.long)
        x_num = torch.tensor([[input_dict['year']]], dtype=torch.float32)
        x_num = torch.tensor(scaler.transform(x_num), dtype=torch.float32)
        model.eval()
        with torch.no_grad():
            logits = model(x_cat.to(device), x_num.to(device))
            prob = torch.softmax(logits, dim=1).cpu().numpy().flatten()
        try:
            win_idx = target_le.transform([driver])[0]
            prob_value = prob[win_idx]
            all_results.append((driver, team, prob_value))
        except Exception as e:
            continue
    if not all_results:
        return "該年參賽車手無法進行預測"
    all_results.sort(key=lambda x: x[2], reverse=True)
    text = f"年份：{year} 場地：{grand_prix}\n\n預測冠軍機率排行（前10名）：\n\n"
    flag_overconfident = False
    for i, (driver, team, prob_value) in enumerate(all_results[:10], 1):
        text += f"{i}. {driver}（{team}）：{prob_value:.2%}\n"
        if prob_value > 0.98:
            flag_overconfident = True
    if flag_overconfident:
        text += "\n⚠️ 機率極高可能代表資料極不平衡或模型未見過此組合，僅供參考。\n"
    return text

# ========== 13. Gradio 追加訓練 ==========
def gradio_train(new_data):
    try:
        global df, model, trainloader, testloader
        df_new = pd.DataFrame(new_data)
        for col in ['year', 'Grand Prix', 'Winner', 'Team', 'Driver', 'Nationality']:
            df_new[col] = df_new[col].astype(str).str.strip()
        df_new = df_new.dropna().reset_index(drop=True)
        for col, le in label_encoders.items():
            df_new[col] = le.transform(df_new[col])
        df_new[num_cols] = scaler.transform(df_new[num_cols])
        df_new['target'] = target_le.transform(df_new['Winner'])
        df = pd.concat([df, df_new], ignore_index=True)
        # 重新製作 Weighted Sampler
        class_sample_count = np.array([len(np.where(df['target']==t)[0]) for t in np.unique(df['target'])])
        weight = 1. / class_sample_count
        samples_weight = np.array([weight[t] for t in df['target']])
        samples_weight = torch.from_numpy(samples_weight).float()
        sampler = WeightedRandomSampler(samples_weight, len(samples_weight))
        trainset = F1Dataset(df)
        trainloader = DataLoader(trainset, batch_size=batch_size, sampler=sampler)
        train_model(model, trainloader, n_epoch=10, lr=0.0005, patience=2)
        model.load_state_dict(torch.load('./data/f1_dnn_embedding.pth'))
        return "追加訓練完成！模型已自動更新。"
    except Exception as e:
        return f"訓練失敗：{str(e)}"

# ========== 14. Gradio UI ==========
with gr.Blocks() as demo:
    gr.Markdown("## F1 冠軍預測互動系統")
    with gr.Tab("冠軍預測"):
        year_input = gr.Dropdown(label="年份", choices=all_years, value=all_years[-1])
        grand_prix = gr.Dropdown(label="Grand Prix 場地", choices=all_grandprix, value=all_grandprix[0])
        predict_btn = gr.Button("預測該場冠軍")
        output = gr.Textbox(label="預測結果")
        predict_btn.click(
            gradio_predict, 
            inputs=[year_input, grand_prix],
            outputs=output
        )
    with gr.Tab("上傳新資料追加訓練"):
        gr.Markdown("上傳新資料(DataFrame結構)：year, Grand Prix, Winner, Team, Driver, Nationality")
        data_upload = gr.Dataframe(headers=['year','Grand Prix','Winner','Team','Driver','Nationality'])
        train_btn = gr.Button("追加訓練")
        train_output = gr.Textbox(label="追加訓練狀態")
        train_btn.click(gradio_train, inputs=[data_upload], outputs=train_output)
demo.launch()



Epoch 1/50 | Train Loss: 4.3469
Epoch 2/50 | Train Loss: 4.2218
Epoch 3/50 | Train Loss: 4.0735
Epoch 4/50 | Train Loss: 3.9587
Epoch 5/50 | Train Loss: 3.7852
Epoch 6/50 | Train Loss: 3.6538
Epoch 7/50 | Train Loss: 3.5133
Epoch 8/50 | Train Loss: 3.3259
Epoch 9/50 | Train Loss: 3.1338
Epoch 10/50 | Train Loss: 2.9106
Epoch 11/50 | Train Loss: 2.7454
Epoch 12/50 | Train Loss: 2.5436
Epoch 13/50 | Train Loss: 2.3540
Epoch 14/50 | Train Loss: 2.1419
Epoch 15/50 | Train Loss: 2.0282
Epoch 16/50 | Train Loss: 1.8849
Epoch 17/50 | Train Loss: 1.6895
Epoch 18/50 | Train Loss: 1.6058
Epoch 19/50 | Train Loss: 1.4960
Epoch 20/50 | Train Loss: 1.3808
Epoch 21/50 | Train Loss: 1.3210
Epoch 22/50 | Train Loss: 1.2843
Epoch 23/50 | Train Loss: 1.2629
Epoch 24/50 | Train Loss: 1.1928
Epoch 25/50 | Train Loss: 1.1493
Epoch 26/50 | Train Loss: 1.1223
Epoch 27/50 | Train Loss: 1.1200
Epoch 28/50 | Train Loss: 1.0889
Epoch 29/50 | Train Loss: 1.0827
Epoch 30/50 | Train Loss: 1.0780
Epoch 31/50 | Train

c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\

Epoch 50/50 | Train Loss: 0.9578
Test Accuracy: 0.9814
                       precision    recall  f1-score   support

         Alain  Prost       1.00      1.00      1.00        14
           Alan Jones       1.00      1.00      1.00         4
      Alberto  Ascari       1.00      1.00      1.00         1
        Ayrton  Senna       1.00      1.00      1.00         9
    Carlos  Reutemann       1.00      1.00      1.00         3
        Carlos  Sainz       1.00      1.00      1.00         1
      Clay  Regazzoni       1.00      1.00      1.00         2
          Damon  Hill       1.00      1.00      1.00         6
    Daniel  Ricciardo       1.00      1.00      1.00         1
     David  Coulthard       1.00      1.00      1.00         4
       Didier  Pironi       1.00      0.50      0.67         2
     Elio  de Angelis       0.00      0.00      0.00         1
  Emerson  Fittipaldi       1.00      1.00      1.00         3
        Felipe  Massa       1.00      1.00      1.00         2

c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with f